# Notebook 03: Output-Preserving Adversarial Attacks (Experiment 2)

**Novel contribution** (not in Li et al. 2025).

Craft perturbations that **maximally disrupt SAE feature activations** while keeping
the model's next-token distribution nearly identical.

This is the most dangerous scenario for safety monitoring: the model's surface
behaviour is indistinguishable from normal, but the interpretability tool sees
something completely different.

**Attack formulation (Lagrangian relaxation):**
```
min  -cosine_sim(SAE_clean, SAE_perturbed)
      + λ · max(0, KL(p_clean ‖ p_perturbed) − δ)
s.t. ‖δ_emb‖_∞ ≤ ε
```

Results → `results/exp2_output_preserving.json`

In [ ]:
!pip install sae-lens==4.3.1 transformers==4.44.2 \
    datasets==2.20.0 tqdm==4.66.4 scipy==1.13.1 -q

In [ ]:
import os, sys, json, random, time
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/rajo69/Adversarial-Robustness-of-SAE-Interpretations"
REPO_DIR = "/content/SAE_Adversarial_Project"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR  = Path("figures");  FIGURES_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

In [ ]:
from transformer_lens import HookedTransformer
from sae_lens import SAE

try:
    model = HookedTransformer.from_pretrained("gemma-2-2b", device=DEVICE)
    MODEL_NAME = "gemma-2-2b"; SAE_RELEASE = "gemma-scope-2b-pt-res"
    SAE_ID_TMPL = "layer_{layer}/width_16k/average_l0_82"
    HOOK_TMPL = "blocks.{layer}.hook_resid_post"; TARGET_LAYER = 12
except Exception as e:
    print(f"Gemma fallback ({e})")
    model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    MODEL_NAME = "gpt2"; SAE_RELEASE = "gpt2-small-res-jb"
    SAE_ID_TMPL = "blocks.{layer}.hook_resid_pre"
    HOOK_TMPL = "blocks.{layer}.hook_resid_pre"; TARGET_LAYER = 8

model.eval()
sae, _, _ = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID_TMPL.format(layer=TARGET_LAYER))
sae = sae.to(DEVICE); sae.eval()
print(f"Model: {MODEL_NAME} | Layer: {TARGET_LAYER}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from src.data import load_eval_set, tokens_to_tensor

eval_set = load_eval_set()
torch.manual_seed(SEED)
idx = sorted(torch.randperm(len(eval_set))[:100].tolist())
exp_set = [eval_set[i] for i in idx]
print(f"Experiment subset: {len(exp_set)} prompts")

In [ ]:
DELTA_KL_VALUES = [0.01, 0.1, 0.5, 1.0]  # output-preservation tolerance
EPSILON         = 0.1                      # fixed L-inf budget (same as Exp 1)
OPA_STEPS       = 40                       # Lagrangian optimisation steps

print(f"Fixed ε={EPSILON}  |  δ_KL values: {DELTA_KL_VALUES}  |  steps={OPA_STEPS}")
print(f"Total OPA runs: {len(DELTA_KL_VALUES)} × {len(exp_set)} = {len(DELTA_KL_VALUES)*len(exp_set)}")

In [ ]:
from src.attacks import output_preserving_attack
from src.metrics import compute_all_metrics

results_per_delta = {}

for delta_kl in DELTA_KL_VALUES:
    print(f"\n{'='*55}\n  δ_KL = {delta_kl}\n{'='*55}")
    prompt_results = []
    hn = HOOK_TMPL.format(layer=TARGET_LAYER)

    for item in tqdm(exp_set, desc=f"δ={delta_kl}"):
        tok = tokens_to_tensor(item["tokens"], device=DEVICE)

        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(tok, names_filter=hn)
            cl_acts = sae.encode(cl_cache[hn])

        pert_emb, info = output_preserving_attack(
            model=model, sae=sae, clean_tokens=tok,
            target_layer=TARGET_LAYER,
            epsilon=EPSILON, delta_kl=delta_kl,
            steps=OPA_STEPS, device=DEVICE,
        )

        with torch.no_grad():
            def _h(v, h): return pert_emb
            pt_logits, pt_cache = model.run_with_cache(
                tok, names_filter=hn, fwd_hooks=[("hook_embed", _h)])
            pt_acts = sae.encode(pt_cache[hn])

        m = compute_all_metrics(cl_acts, pt_acts, cl_logits, pt_logits)
        m["final_kl_info"]  = info["final_kl"]
        m["sae_cos_info"]   = info["final_sae_cosine"]
        m["lambda_final"]   = info["lambda_history"][-1] if info["lambda_history"] else 0.0
        m["eval_idx"]       = item["eval_idx"]
        prompt_results.append(m)

        if DEVICE == "cuda": torch.cuda.empty_cache()

    results_per_delta[str(delta_kl)] = prompt_results
    cos_m = np.mean([m["cosine_similarity"] for m in prompt_results])
    kl_m  = np.mean([m["kl_divergence"]     for m in prompt_results])
    print(f"  SAE cosine={cos_m:.4f}  output KL={kl_m:.4f}")

In [ ]:
# Pareto summary: max SAE disruption achievable at each output-preservation level
pareto = {}
for d_s, res in results_per_delta.items():
    df   = float(d_s)
    cos  = [m["cosine_similarity"]  for m in res]
    kl   = [m["kl_divergence"]      for m in res]
    jac  = [m["jaccard_index"]      for m in res]
    flip = [m["feature_flip_count"] for m in res]
    dis  = [1-c for c in cos]
    # Attack succeeded if: high disruption (>0.3) AND output roughly preserved (KL < 2×target)
    n_ok = sum(1 for d, k in zip(dis, kl) if d > 0.3 and k < df * 2)
    pareto[d_s] = {
        "delta_kl_target":    df,
        "mean_sae_disruption":float(np.mean(dis)),  "std_sae_disruption": float(np.std(dis)),
        "mean_output_kl":     float(np.mean(kl)),   "std_output_kl":      float(np.std(kl)),
        "mean_jaccard":       float(np.mean(jac)),
        "mean_flip_count":    float(np.mean(flip)),
        "n_succeeded":        int(n_ok),
        "pct_succeeded":      float(n_ok / len(res) * 100),
    }
    p = pareto[d_s]
    print(f"δ={d_s}: disruption={p['mean_sae_disruption']:.4f}  "
          f"output KL={p['mean_output_kl']:.4f}  "
          f"success={p['pct_succeeded']:.1f}%")

In [ ]:
output = {
    "experiment": "output_preserving_attack", "model": MODEL_NAME,
    "sae_release": SAE_RELEASE, "target_layer": TARGET_LAYER,
    "epsilon": EPSILON, "delta_kl_values": DELTA_KL_VALUES,
    "opa_steps": OPA_STEPS, "n_prompts": len(exp_set), "seed": SEED,
    "pareto_summary": pareto,
    "per_prompt": {d_s: res for d_s, res in results_per_delta.items()},
}
p = RESULTS_DIR / "exp2_output_preserving.json"
p.write_text(json.dumps(output, indent=2))
print(f"Saved: {p}")

In [ ]:
from src.plot_config import apply_style, save_fig, COLORS

apply_style()
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    f"Exp 2: Output-Preserving Attack\n{MODEL_NAME}, ε={EPSILON}, Layer {TARGET_LAYER}",
    fontsize=13)

d_keys    = list(pareto.keys())
kl_means  = [pareto[k]["mean_output_kl"]      for k in d_keys]
kl_stds   = [pareto[k]["std_output_kl"]       for k in d_keys]
dis_means = [pareto[k]["mean_sae_disruption"] for k in d_keys]
dis_stds  = [pareto[k]["std_sae_disruption"]  for k in d_keys]

# 1) Pareto frontier
ax = axes[0]
ax.errorbar(kl_means, dis_means, xerr=kl_stds, yerr=dis_stds,
            fmt='o-', color=COLORS["adversarial"], capsize=5)
for i, d_s in enumerate(d_keys):
    ax.annotate(f"δ={d_s}", (kl_means[i], dis_means[i]),
                xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Mean Output KL Divergence (↓ = output more preserved)")
ax.set_ylabel("Mean SAE Disruption (1 − cosine sim)")
ax.set_title("Pareto Frontier:\nSAE Disruption vs Output Preservation")

# 2) Per-prompt scatter at the most lenient delta
ax = axes[1]
most_lenient = str(DELTA_KL_VALUES[-1])
res = results_per_delta[most_lenient]
x = [m["kl_divergence"]  for m in res]
y = [m["sae_disruption"] for m in res]
ax.scatter(x, y, alpha=0.6, s=30, color=COLORS["adversarial"], label=f"δ={most_lenient}")
ax.axvline(DELTA_KL_VALUES[-1], color="black", ls="--", lw=1,
           label=f"δ_KL target = {DELTA_KL_VALUES[-1]}")
ax.set_xlabel("Output KL Divergence")
ax.set_ylabel("SAE Disruption (1 − cosine sim)")
ax.set_title(f"Per-Prompt Results (δ={most_lenient})\nTop-left = attack succeeded")
ax.legend()

plt.tight_layout()
fp = FIGURES_DIR / "exp2_output_preserving.png"
save_fig(fig, str(fp)); plt.show()
print(f"Saved: {fp}")

## ✓ Experiment 2 complete

**Record in `progress.md`:** Pareto frontier shape, % of prompts where attack succeeded,
max SAE disruption at each δ_KL, how this compares to unconstrained PGD (Exp 1).

Next: `04_layerwise_sensitivity.ipynb`.